### Import

In [1]:
import os
import gc
import time
import pickle 
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.auto import tqdm
import matplotlib.ticker as ticker
from matplotlib import pyplot as plt 
import pyarrow as pa
import pyarrow.parquet as pq
from pyarrow.parquet import ParquetFile
from sklearn.preprocessing import OneHotEncoder
pd.set_option('display.max_columns', 500)

### General parameters

In [2]:
path_data_mimiciii = "Prediction/MIMICIII/Data/EHR/"
path_data_mimiciv  = "Prediction/MIMICIV/Data/EHR/"

path_data = "Prediction/eICU/Data/"
eicu_path = "PATH TO DATA/eICU/"

race_path_mimiciii = "Extraction/MIMICIII/Data/csvExtract/"
race_path_mimiciv  = "Extraction/MIMICIV/Data/csvExtract/"
race_path_eicu     = "Extraction/eICU/Data/csvExtract/"

### Reading Data

In [4]:
df_mimiciii = pd.read_csv(path_data_mimiciii + '0h_to_24h_data.csv', low_memory=False, index_col=False)
df_mimiciii.head(2)

In [5]:
print(df_mimiciii.ICUSTAY_ID.nunique())
print(df_mimiciii.shape)

59653
(1364554, 543)


In [6]:
df_mimiciv = pd.read_csv(path_data_mimiciv + '0h_to_24h_data.csv', low_memory=False, index_col=False)
df_mimiciv.head(2)

In [7]:
print(df_mimiciv.stay_id.nunique())
print(df_mimiciv.shape)

71344
(1638420, 526)


In [8]:
df_eicu = pd.read_parquet(path_data + 'eICU_raw.parquet' , engine='pyarrow')
df_eicu.head(2)

In [9]:
print(df_eicu.patientunitstayid.nunique())
print(df_eicu.shape)

132837
(3188088, 387)


### Drop Repeated Rows

In [10]:
all_columns = list(df_mimiciii.columns)
remove_columns = [col for col in all_columns if ('_tslm' in col) or ('_diff' in col)]

text_columns = ['Note', 'Discharge_Note', 'Heart Rhythm', 'Ventilator Mode', 'Ventilator Type', 'Ventilator']

remove_columns.extend(text_columns)
df_mimiciii.drop(remove_columns, axis=1, inplace=True)
df_mimiciii = df_mimiciii.drop_duplicates()

print(df_mimiciii.ICUSTAY_ID.nunique())
print(df_mimiciii.shape)

59653
(1333367, 287)


In [11]:
all_columns = list(df_mimiciv.columns)
remove_columns = [col for col in all_columns if ('_tslm' in col) or ('_diff' in col)]

text_columns = ['cxr_image', 'cxr_lung', 'cxr_note', 'radiology_note', 'discharge_note', 
                'Heart Rhythm', 'Ventilator Mode', 'Ventilator Type', 'Ventilator']

remove_columns.extend(text_columns)
df_mimiciv.drop(remove_columns, axis=1, inplace=True)
df_mimiciv = df_mimiciv.drop_duplicates()

print(df_mimiciv.stay_id.nunique())
print(df_mimiciv.shape)

71344
(1631022, 277)


### eICU Information

In [13]:
eicu_patients = pd.read_csv(eicu_path + "patient.csv")
eicu_patients = eicu_patients[['patientunitstayid', 'ethnicity', 'hospitalid']]
eicu_patients = eicu_patients[eicu_patients.patientunitstayid.isin(df_eicu.patientunitstayid.unique())]
eicu_apache_adm = df_eicu[['patientunitstayid', 'apacheadmissioncategory']].drop_duplicates()
eicu_patients = eicu_patients.merge(eicu_apache_adm, on='patientunitstayid')

eicu_patients.loc[eicu_patients.ethnicity.isnull(), 'ethnicity'] = 'Other/Unknown'
eicu_patients.loc[eicu_patients.ethnicity == 'Caucasian', 'ethnicity'] = 'White'
eicu_patients = eicu_patients.rename(columns={"patientunitstayid": "stay_id"})
eicu_patients.head(2)

### Fix Age

In [14]:
df_mimiciii.loc[df_mimiciii['AGE'] >= 95, 'AGE'] = 95
df_mimiciii = df_mimiciii[df_mimiciii.AGE > 16]

df_mimiciv.loc[df_mimiciv['age'] >= 95, 'age'] = 95
df_mimiciv = df_mimiciv[df_mimiciv.age > 16]

### Take first 24 hour LoS

In [16]:
max_rows = df_mimiciii.groupby('ICUSTAY_ID').count()
observation_window = max_rows.Bins.max()
print(observation_window)

24


In [17]:
max_rows = df_mimiciv.groupby('stay_id').count()
observation_window = max_rows.Bins.max()
print(observation_window)

24


In [20]:
max_rows = df_eicu.groupby('patientunitstayid').count()
observation_window = max_rows.Bins.max()
print(observation_window)

24


In [21]:
df_mimiciii = df_mimiciii.groupby('ICUSTAY_ID').head(observation_window).reset_index(drop=True)
df_mimiciv  = df_mimiciv.groupby('stay_id').head(observation_window).reset_index(drop=True)
df_eicu     = df_eicu.groupby('patientunitstayid').head(observation_window).reset_index(drop=True)

### Find Similar Variables

In [22]:
mimiciii_columns = ['ICUSTAY_ID', 'Bins', 'AGE', 'GENDER', 'ETHNICITY', 'Weight', 'Height',
                    'Heart Rate', 'SpO2', 'SvO2', 'Oxygen Saturation', 'Respiratory Rate',
                    'Respiratory Rate (Total)', 'Temperature', 'Non Invasive Blood Pressure mean',
                    'Non Invasive Blood Pressure systolic', 'Non Invasive Blood Pressure diastolic',
                    'Arterial Blood Pressure mean', 'Arterial Blood Pressure systolic', 
                    'Arterial Blood Pressure diastolic',  'Glucose', 'Creatinine', 'Base Excess', 'BUN', 
                    'Anion Gap', 'Bicarbonate', 'Lactate', 'Lactate Dehydrogenase (LD)', 'Hemoglobin', 
                    'Hematocrit', 'pH',  'Bilirubin, Direct', 'Bilirubin, Total', 'pO2', 'pCO2', 'PT', 'PTT', 
                    'INR(PT)', 'AST', 'ALT', 'White Blood Cells', 'WBC', 'Red Blood Cells', 'RDW', 'Platelet Count', 
                    'MCV', 'MCH', 'MCHC', 'Potassium', 'Sodium', 'Chloride', 'Magnesium',
                    'Phosphate', 'Alkaline Phosphatase', 'Calcium, Total',
                    'Cholesterol, Total', 'Differential-Bands', 'Differential-Monos', 
                    'Differential-Lymphs', 'Differential-Eos', 'Differential-Basos', 'Mean Airway Pressure',
                    'Plateau Pressure', 'CVP', 'Amylase', 'Albumin', 'Fibrinogen', 'Triglycerides',
                    'Transferrin', 'Ferritin', 'Troponin T', 'FiO2', 'PEEP', 'PEEP (Set)', 
                    'Tidal Volume', 'Tidal Volume (Set)', 'GCS Total', 'GCS - Eye Opening',  
                    'GCS - Verbal Response',  'GCS - Motor Response' , 'Pain Present', 'Richmond-RAS Scale', 
                    'Pain Level',  'Delirium assessment', 'UrineOutput_IO', 'Ventilation Rate',
                    'PaO2/FiO2', 'SIRS', 'Shock_Index', 'SOFA', 'SAPSII', 'OASIS',
                    'Norepinephrine_PRC', 'Fentanyl_PRC', 'Antibiotic_PRC', 'Heparin_PRC', 'Insulin_PRC', 
                    'Propofol_PRC', 'Phenylephrine_PRC', 'Vasopressin_PRC', 'Epinephrine_PRC', 'Pantoprazole_PRC',
                    'ICU_EXPIRE_FLAG'] 

In [23]:
mimiciv_columns = ['stay_id', 'Bins', 'age', 'gender', 'race', 'Weight', 'Height',
                   'Heart Rate', 'SpO2', 'SvO2', 'Oxygen Saturation', 'Respiratory Rate',
                   'Respiratory Rate (Total)', 'Temperature', 'Non Invasive Blood Pressure mean', 
                   'Non Invasive Blood Pressure systolic', 'Non Invasive Blood Pressure diastolic', 
                   'Arterial Blood Pressure mean', 'Arterial Blood Pressure systolic', 
                   'Arterial Blood Pressure diastolic',  'Glucose', 'Creatinine', 'Base Excess', 'BUN', 
                   'Anion Gap', 'Bicarbonate',  'Lactate', 'Lactate Dehydrogenase(LDH)', 'Hemoglobin', 
                   'Hematocrit', 'pH', 'Bilirubin, Direct', 'Bilirubin, Total', 'pO2', 'pCO2', 'PT', 'PTT',
                   'INR(PT)', 'AST', 'ALT', 'White Blood Cells', 'WBC', 'Red Blood Cells', 'RDW', 'Platelet Count', 
                   'MCV', 'MCH', 'MCHC', 'Potassium', 'Sodium', 'Chloride', 'Magnesium', 
                   'Phosphate', 'Alkaline Phosphate', 'Calcium, Total',
                   'Cholesterol, Total', 'Differential-Bands', 'Differential-Monos', 
                   'Differential-Lymphs', 'Differential-Eos', 'Differential-Basos', 'Mean Airway Pressure', 
                   'Plateau Pressure', 'Central Venous Pressure', 'Amylase', 'Albumin', 'Fibrinogen', 'Triglyceride',
                   'Transferrin', 'Ferritin', 'Troponin T', 'FiO2', 'PEEP',  'PEEP (Set)', 
                   'Tidal Volume', 'Tidal Volume (Set)', 'Total GCS', 'GCS - Eye Opening', 
                   'GCS - Verbal Response', 'GCS - Motor Response', 'Pain Present', 'Richmond-RAS Scale', 
                   'Pain Level', 'Delirium assessment', 'UrineOutput_IO', 'Ventilation Rate',
                   'PaO2/FiO2', 'SIRS', 'Shock_Index', 'SOFA', 'SAPSII', 'OASIS',
                   'Norepinephrine_PRC', 'Fentanyl_PRC', 'Antibiotic_PRC', 'Heparin_PRC', 'Insulin_PRC', 
                   'Propofol_PRC', 'Phenylephrine_PRC', 'Vasopressin_PRC', 'Epinephrine_PRC', 'Pantoprazole_PRC',
                   'icu_expire_flag']

In [24]:
eICU_columns = ['patientunitstayid', 'Bins', 'age', 'gender', 'ethnicity', 'admissionweight', 'admissionheight',
                'Heart Rate', 'SpO2', 'SVO2', 'O2 Saturation', 'Respiratory Rate', 
                'Total Respiratory Rate', 'Temperature (C)', 'Non-Invasive BP Mean', 'Non-Invasive BP Systolic',
                'Non-Invasive BP Diastolic', 'Invasive BP Mean', 'Invasive BP Systolic', 
                'Invasive BP Diastolic', 'Glucose', 'creatinine', 'Base Excess', 'BUN', 
                'anion gap',  'Bicarbonate',  'lactate', 'LDH', 'Hgb', 
                'Hct', 'pH', 'direct bilirubin', 'total bilirubin', 'paO2', 'paCO2', 'PT', 'PTT',
                'PT - INR', 'AST (SGOT)', 'ALT (SGPT)', 'WBC x 1000', 'RBC', 'RDW', 'platelets x 1000', 
                'MCV', 'MCH', 'MCHC', 'potassium', 'sodium', 'chloride', 'magnesium',
                'phosphate', 'alkaline phos.', 'calcium', 
                'total cholesterol', '-bands', '-monos', 
                '-lymphs', '-eos', '-basos', 'Mean Airway Pressure', 
                'Plateau Pressure', 'CVP', 'amylase', 'albumin', 'fibrinogen', 'triglycerides', 
                'transferrin', 'Ferritin', 'troponin - T', 'FiO2', 'FiO2 (Set)', 'PEEP', 'PEEP (Set)', 
                'Tidal Volume', 'Tidal Volume (Set)', 'GCS Total', 'Eyes', 
                'Verbal','Motor', 'Pain Present', 'RASS', 
                'Pain Score',  'Symptoms of Delirium Present', 'Urine_IO',  'Vent Rate', 'Vent Rate (Set)',
                'PaO2/FiO2', 'SIRS', 'Shock_Index', 'SOFA', 'SAPSII', 'OASIS',
                'Norepinephrine_PRC', 'Fentanyl_PRC', 'Antibiotic_PRC', 'Heparin_PRC', 'Insulin_PRC', 
                'Propofol_PRC', 'Phenylephrine_PRC', 'Vasopressin_PRC', 'Epinephrine_PRC', 'Pantoprazole_PRC',
                'unitdischargestatus']

In [25]:
mimiciii_columns_all = []

for i in mimiciii_columns:

    if i in list(df_mimiciii.columns):
        mimiciii_columns_all.append(i)

    temp_col = i + '_ind'
    if temp_col in list(df_mimiciii.columns):
        mimiciii_columns_all.append(temp_col)

In [26]:
mimiciv_columns_all = []

for i in mimiciv_columns:

    if i in list(df_mimiciv.columns):
        mimiciv_columns_all.append(i)

    temp_col = i + '_ind'
    if temp_col in list(df_mimiciv.columns):
        mimiciv_columns_all.append(temp_col)

In [27]:
eICU_columns_all = []

for i in eICU_columns:

    if i in list(df_eicu.columns):
        eICU_columns_all.append(i)

    temp_col = i + '_ind'
    if temp_col in list(df_eicu.columns):
        eICU_columns_all.append(temp_col)

### Select Variables

In [28]:
df_mimiciii = df_mimiciii[mimiciii_columns_all]
df_mimiciv  = df_mimiciv[mimiciv_columns_all]
df_eicu  = df_eicu[eICU_columns_all]

In [29]:
wbc_condition_iii = (df_mimiciii['White Blood Cells'].isnull()) & (df_mimiciii['WBC'].notnull())
df_mimiciii.loc[wbc_condition_iii, 'White Blood Cells'] = df_mimiciii['WBC']
df_mimiciii.loc[wbc_condition_iii, 'White Blood Cells_ind'] = df_mimiciii['WBC_ind']

df_mimiciii.drop(columns=['WBC', 'WBC_ind'], inplace=True)

In [30]:
wbc_condition_iv = (df_mimiciv['White Blood Cells'].isnull()) & (df_mimiciv['WBC'].notnull())
df_mimiciv.loc[wbc_condition_iv, 'White Blood Cells'] = df_mimiciv['WBC']
df_mimiciv.loc[wbc_condition_iv, 'White Blood Cells_ind'] = df_mimiciv['WBC_ind']

df_mimiciv.drop(columns=['WBC', 'WBC_ind'], inplace=True)

In [31]:
fio2_condition = (df_eicu['FiO2'].isnull()) & (df_eicu['FiO2 (Set)'].notnull())
df_eicu.loc[fio2_condition, 'FiO2'] = df_eicu['FiO2 (Set)']
df_eicu.loc[fio2_condition, 'FiO2_ind'] = df_eicu['FiO2 (Set)_ind']

vent_condition = (df_eicu['Vent Rate'].isnull()) & (df_eicu['Vent Rate (Set)'].notnull())
df_eicu.loc[vent_condition, 'Vent Rate'] = df_eicu['Vent Rate (Set)']
df_eicu.loc[vent_condition, 'Vent Rate_ind'] = df_eicu['Vent Rate (Set)_ind']

df_eicu.drop(columns=['FiO2 (Set)', 'FiO2 (Set)_ind', 'Vent Rate (Set)', 'Vent Rate (Set)_ind'], inplace=True)

### Unifying Column Names

In [32]:
df_mimiciii.columns = list(df_mimiciv.columns)
df_eicu.columns = list(df_mimiciv.columns)

### Remove Missing Gender

In [33]:
df_eicu = df_eicu[df_eicu.gender != 0]
df_eicu = df_eicu[df_eicu.age.notnull()]

### Fixing Race

In [34]:
with open(race_path_mimiciii + 'race_dictionary.pkl', 'rb') as f:
    race_dictionary_mimiciii = pickle.load(f)

In [35]:
with open(race_path_mimiciv + 'race_dictionary.pkl', 'rb') as f:
    race_dictionary_mimiciv = pickle.load(f)

In [36]:
with open(race_path_eicu + 'ethnicity_dictionary.pkl', 'rb') as f:
    race_dictionary_eicu = pickle.load(f)

In [37]:
general_ethnicity_mapping_mimiciii = {
    
    'WHITE': 'White',
    'WHITE - RUSSIAN': 'White',
    'WHITE - BRAZILIAN': 'White',
    'WHITE - OTHER EUROPEAN': 'White',
    'WHITE - EASTERN EUROPEAN': 'White',
    'PORTUGUESE': 'White',
    
    'UNABLE TO OBTAIN': 'Other/Unknown',
    'UNKNOWN/NOT SPECIFIED': 'Other/Unknown',
    'PATIENT DECLINED TO ANSWER': 'Other/Unknown',
    'OTHER': 'Other/Unknown',
    'MIDDLE EASTERN': 'Other/Unknown',
    'CARIBBEAN ISLAND': 'Other/Unknown',
    'MULTI RACE ETHNICITY': 'Other/Unknown',
    'NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER': 'Other/Unknown',
    
    'ASIAN': 'Asian',
    'ASIAN - THAI': 'Asian',
    'ASIAN - OTHER': 'Asian',
    'ASIAN - KOREAN': 'Asian',
    'ASIAN - CHINESE': 'Asian',
    'ASIAN - FILIPINO': 'Asian',
    'ASIAN - JAPANESE': 'Asian',
    'ASIAN - CAMBODIAN': 'Asian',
    'ASIAN - VIETNAMESE': 'Asian',
    'ASIAN - ASIAN INDIAN': 'Asian',
    
    'AMERICAN INDIAN/ALASKA NATIVE': 'Native American',
    'AMERICAN INDIAN/ALASKA NATIVE FEDERALLY RECOGNIZED TRIBE': 'Native American',
    
    'BLACK/AFRICAN': 'African American',
    'BLACK/HAITIAN': 'African American',
    'BLACK/CAPE VERDEAN': 'African American',
    'BLACK/AFRICAN AMERICAN': 'African American',
    
    'SOUTH AMERICAN': 'Hispanic',
    'HISPANIC OR LATINO': 'Hispanic',
    'HISPANIC/LATINO - CUBAN': 'Hispanic',
    'HISPANIC/LATINO - MEXICAN': 'Hispanic',
    'HISPANIC/LATINO - HONDURAN': 'Hispanic',
    'HISPANIC/LATINO - DOMINICAN': 'Hispanic',
    'HISPANIC/LATINO - COLOMBIAN': 'Hispanic',
    'HISPANIC/LATINO - SALVADORAN': 'Hispanic',
    'HISPANIC/LATINO - GUATEMALAN': 'Hispanic',
    'HISPANIC/LATINO - PUERTO RICAN': 'Hispanic',
    'HISPANIC/LATINO - CENTRAL AMERICAN (OTHER)': 'Hispanic'}

In [38]:
general_ethnicity_mapping_mimiciv = {
    
    'WHITE': 'White',
    'WHITE - RUSSIAN': 'White',
    'WHITE - BRAZILIAN': 'White',
    'WHITE - OTHER EUROPEAN': 'White',
    'WHITE - EASTERN EUROPEAN': 'White',
    'PORTUGUESE': 'White',
    
    'UNKNOWN': 'Other/Unknown',
    'UNABLE TO OBTAIN': 'Other/Unknown',
    'PATIENT DECLINED TO ANSWER': 'Other/Unknown',
    'OTHER': 'Other/Unknown',
    'MIDDLE EASTERN': 'Other/Unknown',
    'MULTIPLE RACE/ETHNICITY': 'Other/Unknown',
    'NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER': 'Other/Unknown',
    'AMERICAN INDIAN/ALASKA NATIVE': 'Other/Unknown',
    
    'ASIAN': 'Asian',
    'ASIAN - CHINESE': 'Asian',
    'ASIAN - SOUTH EAST ASIAN': 'Asian',
    'ASIAN - KOREAN': 'Asian',
    'ASIAN - ASIAN INDIAN': 'Asian',
    
    'BLACK/AFRICAN': 'African American',
    'BLACK/CAPE VERDEAN': 'African American',
    'BLACK/AFRICAN AMERICAN': 'African American',
    'BLACK/CARIBBEAN ISLAND': 'African American',
    
    'SOUTH AMERICAN': 'Hispanic',
    'HISPANIC OR LATINO': 'Hispanic',
    'HISPANIC/LATINO - CUBAN': 'Hispanic',
    'HISPANIC/LATINO - MEXICAN': 'Hispanic',
    'HISPANIC/LATINO - HONDURAN': 'Hispanic',
    'HISPANIC/LATINO - COLUMBIAN': 'Hispanic',
    'HISPANIC/LATINO - DOMINICAN': 'Hispanic',
    'HISPANIC/LATINO - SALVADORAN': 'Hispanic',
    'HISPANIC/LATINO - GUATEMALAN': 'Hispanic',
    'HISPANIC/LATINO - PUERTO RICAN': 'Hispanic',
    'HISPANIC/LATINO - CENTRAL AMERICAN': 'Hispanic'}

In [39]:
def replace_ethnicity_with_names(df, ethnicity_dict):
    
    inv_ethnicity_dict = {v: k for k, v in ethnicity_dict.items()}
    df['race'] = df['race'].map(inv_ethnicity_dict)
    
    return df

In [40]:
def categorize_ethnicity(df, new_mapping):
    
    df['race'] = df['race'].map(new_mapping)
    
    return df

In [41]:
df_mimiciii = replace_ethnicity_with_names(df_mimiciii, race_dictionary_mimiciii)
df_mimiciii = categorize_ethnicity(df_mimiciii, general_ethnicity_mapping_mimiciii)

df_mimiciv = replace_ethnicity_with_names(df_mimiciv, race_dictionary_mimiciv)
df_mimiciv = categorize_ethnicity(df_mimiciv, general_ethnicity_mapping_mimiciv)

In [42]:
df_eicu = replace_ethnicity_with_names(df_eicu, race_dictionary_eicu)

In [43]:
df_eicu.loc[df_eicu.race == 'nodx', 'race'] = 'Other/Unknown'
df_eicu.loc[df_eicu.race == 'Caucasian', 'race'] = 'White'

### Encode using 3 bits

In [44]:
all_races = sorted(set(df_mimiciii['race']).union(set(df_mimiciv['race'])).union(set(df_eicu['race'])))
race_to_bits = {race: format(i, '03b') for i, race in enumerate(all_races)}

def encode_race(race):
    return list(map(int, race_to_bits[race]))

df_mimiciii_encoded = df_mimiciii['race'].apply(encode_race).apply(pd.Series)
df_mimiciv_encoded  = df_mimiciv['race'].apply(encode_race).apply(pd.Series)
df_eicu_encoded = df_eicu['race'].apply(encode_race).apply(pd.Series)

df_mimiciii_encoded.columns = [f'race_bit_{i}' for i in range(3)]
df_mimiciv_encoded.columns  = [f'race_bit_{i}' for i in range(3)]
df_eicu_encoded.columns = [f'race_bit_{i}' for i in range(3)]

df_mimiciii_final = pd.concat([df_mimiciii, df_mimiciii_encoded], axis=1)
df_mimiciv_final  = pd.concat([df_mimiciv,  df_mimiciv_encoded],  axis=1)
df_eicu_final = pd.concat([df_eicu, df_eicu_encoded], axis=1)

df_mimiciii_final = df_mimiciii_final.drop('race', axis=1)
df_mimiciv_final  = df_mimiciv_final.drop('race', axis=1)
df_eicu_final = df_eicu_final.drop('race', axis=1)

### One Hot Encoding

In [45]:
# all_races = sorted(set(df_mimiciii['race']).union(set(df_mimiciv['race'])).union(set(df_eicu['race'])))
# encoder = OneHotEncoder(categories=[all_races], sparse=False)

# encoded_mimiciii = encoder.fit_transform(df_mimiciii[['race']])
# encoded_mimiciv  = encoder.transform(df_mimiciv[['race']])
# encoded_eicu = encoder.transform(df_eicu[['race']])

# encoded_mimiciii_df = pd.DataFrame(encoded_mimiciii, columns=encoder.categories_[0])
# encoded_mimiciv_df  = pd.DataFrame(encoded_mimiciv,  columns=encoder.categories_[0])
# encoded_eicu_df = pd.DataFrame(encoded_eicu, columns=encoder.categories_[0])

# encoded_mimiciii_df = encoded_mimiciii_df.add_prefix('race_')
# encoded_mimiciv_df  = encoded_mimiciv_df.add_prefix('race_')
# encoded_eicu_df = encoded_eicu_df.add_prefix('race_')

# df_mimiciii_final = pd.concat([df_mimiciii, encoded_mimiciii_df], axis=1)
# df_mimiciv_final  = pd.concat([df_mimiciv, encoded_mimiciv_df], axis=1)
# df_eicu_final = pd.concat([df_eicu, encoded_eicu_df], axis=1)

# df_mimiciii_final = df_mimiciii_final.drop('race', axis=1)
# df_mimiciv_final  = df_mimiciv_final.drop('race', axis=1)
# df_eicu_final = df_eicu_final.drop('race', axis=1)

### Outlier Detection

In [46]:
df_mimiciii_final.loc[df_mimiciii_final['Height'] < 70, 'Height'] = 70
df_mimiciv_final.loc[df_mimiciv_final['Height'] < 70, 'Height'] = 70

df_mimiciii_final.loc[df_mimiciii_final['Arterial Blood Pressure systolic'] < 10, 'Arterial Blood Pressure systolic'] = 10
df_mimiciii_final.loc[df_mimiciii_final['Arterial Blood Pressure systolic'] > 350, 'Arterial Blood Pressure systolic'] = 350
df_eicu_final.loc[df_eicu_final['Arterial Blood Pressure systolic'] > 350, 'Arterial Blood Pressure systolic'] = 350

df_eicu_final.loc[df_eicu_final['Lactate Dehydrogenase(LDH)'] > 4000, 'Lactate Dehydrogenase(LDH)'] = 4000

df_mimiciv_final.loc[df_mimiciv_final['MCHC'] > 42, 'MCHC'] = 42
df_eicu_final.loc[df_eicu_final['MCHC'] > 42, 'MCHC'] = 42

df_mimiciii_final.loc[df_mimiciii_final['Differential-Monos'] > 65, 'Differential-Monos'] = 65
df_mimiciv_final.loc[df_mimiciv_final['Differential-Monos'] > 65, 'Differential-Monos'] = 65

df_mimiciii_final.loc[df_mimiciii_final['Triglyceride'] > 1350, 'Triglyceride'] = 1350
df_mimiciv_final.loc[df_mimiciv_final['Triglyceride'] > 1350, 'Triglyceride'] = 1350

df_mimiciii_final.loc[df_mimiciii_final['PaO2/FiO2'] > 100, 'PaO2/FiO2'] = 100
df_mimiciv_final.loc[df_mimiciv_final['PaO2/FiO2'] > 100, 'PaO2/FiO2'] = 100
df_eicu_final.loc[df_eicu_final['PaO2/FiO2'] > 100, 'PaO2/FiO2'] = 100

df_mimiciii_final.loc[df_mimiciii_final['Shock_Index'] < 0, 'Shock_Index'] = 0

### Concatenate MIMIC Data

In [47]:
df_mimic = [df_mimiciii_final, df_mimiciv_final]
df_mimic_final = pd.concat(df_mimic)

In [48]:
df_mimic_final.head()

In [49]:
df_eicu_final.head()

### Save Data

In [50]:
mimic_file_path = os.path.join(path_data, 'mimic.parquet')
mimic_table = pa.Table.from_pandas(df_mimic_final)
pq.write_table(mimic_table, mimic_file_path)

In [51]:
eicu_file_path = os.path.join(path_data, 'eicu.parquet')
eicu_table = pa.Table.from_pandas(df_eicu_final)
pq.write_table(eicu_table, eicu_file_path)

In [52]:
eicu_patients.to_csv(path_data + 'eicu_information.csv', index=False)